# TRACER Legacy Workflow

This notebook documents and runs the legacy TRACER workflow that was the default before this branch's algorithmic work — i.e. the workflow at the merge-base of `optimization/core-refactor` with `feature/relative-stitching` (commit `a2e1f37`). It reproduces the original `tutorials/lung_cancer/run_lung_cancer.py` driver script, stage by stage.

## Stages

| step | name | what it does |
|---|---|---|
| 0 | Load NPMI CSV | read precomputed `lung_cancer_npmi.csv` |
| 1 | **Prune** | Stage 1 — conservative NPMI prune (threshold = −0.05) |
| 2 | **Group** | Stage 2 — annotate unassigned components via kNN graph |
| 3 | **Stitch (initial)** | Stage 3 — stitch entities by ΔC merge score |
| 4 | **Split** | Stage 4 — enforce spatial coherence via kNN, splits disconnected labels |
| 5 | **Stitch (final)** | Stage 5 — re-stitch after Split |
| 6 | **Phase 6 reassignment** | reassign unassigned tx by nearest-entity (no NPMI veto) |

**Note**: Some functions used here have been refactored or renamed on the current branch. The notebook calls the closest-equivalent current API; the resulting partition will not be bit-equivalent to the original `feature/relative-stitching` output, but the qualitative behavior (stages, parameters, column lifecycle) matches.

For a side-by-side narrative of what changed between this legacy workflow and the current production workflow, see `algorithmic_changes.ipynb`.

## Setup

In [ ]:
from __future__ import annotations

import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from scipy.spatial import cKDTree
from torch_geometric.data import Data

from tracer.pruning import prune_transcripts_fast, prune_genes_by_npmi_greedy
from tracer.spatial import (
    annotate_unassigned_components_fast,
    enforce_spatial_coherence_fast,
    reassign_unassigned_to_nearby_entities,
)
from tracer.stitching import apply_stitching_to_transcripts_memory_efficient

# Project paths
PROJECT_DIR = Path("/Users/adeshpa6/1_Projects/01.10_Lab/GENESIS/tutorials/lung_cancer")
DATA_DIR = PROJECT_DIR / "data"
OUT_DIR = PROJECT_DIR / "output"
OUT_DIR.mkdir(parents=True, exist_ok=True)

### Load transcripts and legacy NPMI CSV

The legacy workflow used a precomputed `lung_cancer_npmi.csv` panel (no bootstrap; observation-only NaN handling). For this notebook we'll use either that legacy CSV if present, or fall back to the project parquet file.

In [ ]:
# Pick the parquet — the legacy script used `lung_cancer_df_nuclear_expansion.parquet`.
# Fall back to whatever data parquet exists in <project>/data/.
candidates = sorted(DATA_DIR.glob("*.parquet"))
if not candidates:
    raise FileNotFoundError(f"No .parquet in {DATA_DIR}")
parquet_path = candidates[0]
for c in candidates:
    if "nuclear_expansion" in c.name:
        parquet_path = c
        break

npmi_csv_path = DATA_DIR / "lung_cancer_npmi.csv"

print(f"Transcripts: {parquet_path}")
print(f"NPMI CSV:    {npmi_csv_path}")

t0 = time.time()
df_transcripts = pd.read_parquet(parquet_path)
print(f"Loaded {len(df_transcripts):,} transcripts in {time.time()-t0:.1f}s.")

t0 = time.time()
df_npmi = pd.read_csv(npmi_csv_path)
print(f"Loaded {len(df_npmi):,} NPMI rows in {time.time()-t0:.1f}s.")
print(f"NPMI columns: {list(df_npmi.columns)}")

### Legacy kNN graph builder (`build_graph_fast`)

The legacy driver defined a fast cKDTree-based kNN graph builder inline. Reproduced here verbatim — used by Stage 2 (Group) and Stage 4 (Split).

In [ ]:
def build_graph_fast(
    df,
    *,
    k=10,
    dist_threshold=1.5,
    coord_cols=("x", "y", "z"),
):
    coords = df[list(coord_cols)].to_numpy(dtype=np.float32)
    N = coords.shape[0]

    tree = cKDTree(coords)
    distances, indices = tree.query(coords, k=min(k + 1, N))

    if indices.ndim == 1:
        indices = indices[:, None]
        distances = distances[:, None]
    mask = (distances <= dist_threshold)
    mask[:, 0] = False  # remove self

    src = np.repeat(np.arange(N), mask.sum(axis=1))
    tgt = indices[mask]

    edge_index = torch.from_numpy(np.vstack([src, tgt]).astype(np.int64))

    data = Data(pos=torch.from_numpy(coords), edge_index=edge_index)
    data.gene_name = df["feature_name"].to_numpy().astype(str)
    data.id = df["transcript_id"].to_numpy().astype(str)

    print(f"  kNN edges={edge_index.shape[1]:,} N={N:,} (k≤{k}, d≤{dist_threshold} µm)")
    return data

### Helpers: per-stage state tracking

In [ ]:
def classify(label: str) -> str:
    s = str(label)
    if s in ("DROP", "-1", "nan", "UNASSIGNED"):
        return "unassigned"
    if s.startswith("UNASSIGNED_"):
        return "component"
    if "-" in s:
        return "partial"
    return "cell"


def state(df: pd.DataFrame, col: str) -> dict:
    s = df[col].astype(str)
    types = s.map(classify)
    n_ent = s.groupby(types).nunique().to_dict()
    n_tx = types.value_counts().to_dict()
    return {
        "n_cells": int(n_ent.get("cell", 0)),
        "n_partials": int(n_ent.get("partial", 0)),
        "n_components": int(n_ent.get("component", 0)),
        "tx_cells": int(n_tx.get("cell", 0)),
        "tx_partials": int(n_tx.get("partial", 0)),
        "tx_components": int(n_tx.get("component", 0)),
        "tx_unassigned": int(n_tx.get("unassigned", 0)),
    }


def fmt(s: dict) -> str:
    return (
        f"cells={s['n_cells']:,}/{s['tx_cells']:,}tx, "
        f"partials={s['n_partials']:,}/{s['tx_partials']:,}tx, "
        f"components={s['n_components']:,}/{s['tx_components']:,}tx, "
        f"unassigned={s['tx_unassigned']:,}tx"
    )


stages_log: list[tuple[str, dict]] = []
_input_state = state(df_transcripts.assign(_lbl=df_transcripts["cell_id"].astype(str)), "_lbl")
stages_log.append(("input (cell_id)", _input_state))
print(f"Input state: {fmt(_input_state)}")

## Stage 1 — Prune

Conservative NPMI pruning at `threshold = -0.05`. The legacy threshold was on the NPMI scale (lenient negative — drop a tx only if its gene is *strongly* avoided by the rest of its cell's panel). Modern workflow uses PMI threshold `+ln(1.5)` (strict positive — *require* gene-coherence).

In [ ]:
t0 = time.time()
df_pruned, aux = prune_transcripts_fast(
    df=df_transcripts,
    npmi_df=df_npmi,
    cell_id_col="cell_id",
    gene_col="feature_name",
    threshold=-0.05,           # legacy NPMI threshold
    unassigned_id="-1",
    n_jobs=-1,
    show_progress=False,
)
# Legacy column lifecycle: the prune output is in `tracer_id` (the new
# canonical column). The legacy code wrote to `cell_id_npmi_cons_p2`.
df_pruned["cell_id_npmi_cons_p2"] = df_pruned["tracer_id"].astype(str)
s = state(df_pruned, "cell_id_npmi_cons_p2")
stages_log.append(("after Stage 1 (Prune)", s))
print(f"Stage 1 ({time.time()-t0:.1f}s): {fmt(s)}")

## Stage 2 — Group (annotate unassigned components)

Build a kNN graph (k=8, d≤1.5 µm) over all transcripts and identify connected components among the unassigned tx. Components ≥10 tx survive; smaller ones get demoted. Within each component, gene-set is greedy-pruned at NPMI ≤ -0.1.

Legacy output column: `cell_id_final` (after this stage, holds either an input `cell_id`, a `cell_id_npmi_cons_p2` partial label, or a new `UNASSIGNED_<i>` label).

In [ ]:
t0 = time.time()
df_final = annotate_unassigned_components_fast(
    df_pruned=df_pruned,
    aux=aux,
    build_graph_fn=build_graph_fast,
    prune_fn=prune_genes_by_npmi_greedy,
    coord_cols=("x", "y", "z"),
    k=8,
    dist_threshold=1.5,
    min_comp_size=10,
    npmi_threshold=-0.1,
    entity_col="tracer_id",
    out_col="tracer_id",
    cell_id_col="cell_id",
    gene_col="feature_name",
    transcript_id_col="transcript_id",
    show_progress=False,
)
df_final["cell_id_final"] = df_final["tracer_id"].astype(str)
s = state(df_final, "cell_id_final")
stages_log.append(("after Stage 2 (Group)", s))
print(f"Stage 2 ({time.time()-t0:.1f}s): {fmt(s)}")

## Stage 3 — Stitch (initial)

Run stitch on the post-Group entities. The legacy parameter set:
- `purity_threshold=0.05` — minimum coherence for a merge to be accepted (renamed to `threshold` on the current branch).
- `deltaC_min=0.01` — minimum ΔC improvement.
- `use_3d=True` — 3D stitching (legacy flag; current branch infers from `coord_cols`).
- `dist_threshold=5.0` — Euclidean limit for candidate pairs.

Legacy output column: `cell_id_stitched`.

In [ ]:
t0 = time.time()
df_stitched, _ = apply_stitching_to_transcripts_memory_efficient(
    df_final=df_final,
    aux=aux,
    entity_col="cell_id_final",
    gene_col="feature_name",
    coord_cols=("x", "y", "z"),
    mode="count",
    metric="npmi",
    threshold=0.05,            # legacy `purity_threshold` is now `threshold`
    penalize_simplicity=True,
    deltaC_min=0.01,           # legacy strict ΔC floor (modern uses 0.0)
    dist_threshold=5.0,
    out_col="cell_id_stitched",
    show_progress=False,
    candidate_source="knn",    # legacy used kNN candidates, not grid
)
s = state(df_stitched, "cell_id_stitched")
stages_log.append(("after Stage 3 (Stitch initial)", s))
print(f"Stage 3 ({time.time()-t0:.1f}s): {fmt(s)}")

## Stage 4 — Split (spatial coherence)

Identify any `cell_id_stitched` label whose member transcripts span multiple connected components in a kNN(k=5, d≤5) graph, and split each into separate fragments. Note that the legacy Stage 4 ran AFTER initial stitching (between Stage 3 and Stage 5), not after Stage 1.

Legacy output column: `cell_id_spatial`.

In [ ]:
t0 = time.time()
df_split = enforce_spatial_coherence_fast(
    df_stitched=df_stitched,
    build_graph_fn=build_graph_fast,
    entity_col="cell_id_stitched",
    coord_cols=("x", "y", "z"),
    k=5,
    dist_threshold=5.0,
    out_col="cell_id_spatial",
    show_progress=False,
)
s = state(df_split, "cell_id_spatial")
stages_log.append(("after Stage 4 (Split)", s))
print(f"Stage 4 ({time.time()-t0:.1f}s): {fmt(s)}")

## Stage 5 — Stitch (final)

Re-run stitch on the post-Split labels. The intuition: Stage 4 may have produced fragments that are now better candidates for merging with neighboring entities.

Legacy output column: `cell_id_finetuned`.

In [ ]:
t0 = time.time()
df_finetuned, _ = apply_stitching_to_transcripts_memory_efficient(
    df_final=df_split,
    aux=aux,
    entity_col="cell_id_spatial",
    gene_col="feature_name",
    coord_cols=("x", "y", "z"),
    mode="count",
    metric="npmi",
    threshold=0.05,
    penalize_simplicity=True,
    deltaC_min=0.01,
    dist_threshold=5.0,
    out_col="cell_id_finetuned",
    show_progress=False,
    candidate_source="knn",
)
s = state(df_finetuned, "cell_id_finetuned")
stages_log.append(("after Stage 5 (Stitch final)", s))
print(f"Stage 5 ({time.time()-t0:.1f}s): {fmt(s)}")

## Phase 6 — Reassign unassigned transcripts

Take any tx still labeled `"-1"` after Stage 5 and reassign them to the nearest entity within `dist_threshold=5.0` µm. The legacy version had **no NPMI veto** — it picked the nearest entity regardless of gene compatibility (a known limitation that the modern `reassign_unassigned_grid_pool` addressed by adding the mean-PMI veto).

**Function rename**: legacy `reassign_unassigned_to_nearby_entities_fast` → current branch's `reassign_unassigned_to_nearby_entities` (the `_fast` suffix was dropped after the per-tx loop was vectorized; see commit `e812629`).

Legacy output column: `cell_id_finetuned_2`.

In [ ]:
t0 = time.time()
df_finetuned, n_reassigned, p6_stats = reassign_unassigned_to_nearby_entities(
    df_finetuned,
    entity_summary=None,
    entity_col="cell_id_finetuned",
    gene_col="feature_name",
    coord_cols=("x", "y", "z"),
    out_col="cell_id_finetuned_2",
    dist_threshold=5.0,
    only_partial_component=False,
    show_progress=False,
)
s = state(df_finetuned, "cell_id_finetuned_2")
stages_log.append(("after Phase 6 (FINAL)", s))
print(f"Phase 6 ({time.time()-t0:.1f}s): reassigned={n_reassigned:,}")
print(f"  {fmt(s)}")

## Stage progression table

In [ ]:
rows = []
for name, s in stages_log:
    n_total = s["tx_cells"] + s["tx_partials"] + s["tx_components"] + s["tx_unassigned"]
    n_assigned = s["tx_cells"] + s["tx_partials"] + s["tx_components"]
    pct = 100 * n_assigned / max(n_total, 1)
    rows.append({
        "stage": name,
        "cells": s["n_cells"],
        "partials": s["n_partials"],
        "components": s["n_components"],
        "tx_cells": s["tx_cells"],
        "tx_partials": s["tx_partials"],
        "tx_components": s["tx_components"],
        "tx_unassigned": s["tx_unassigned"],
        "pct_assigned": round(pct, 2),
    })
progression = pd.DataFrame(rows)
print(progression.to_string(index=False))

### Quick QC: per-entity transcript-count quartiles

In [ ]:
final = df_finetuned["cell_id_finetuned_2"].astype(str)
sizes = df_finetuned.loc[final != "-1", "cell_id_finetuned_2"].astype(str).value_counts()
kind_idx = sizes.index.to_series().map(classify)
print("per-entity transcript-count quartiles")
print("-" * 64)
for kind in ("cell", "partial", "component"):
    sub = sizes[kind_idx == kind]
    if len(sub) == 0:
        continue
    q = np.quantile(sub, [0.25, 0.5, 0.75, 0.9])
    print(f"  {kind:<10} n={len(sub):>5,}  Q1={int(q[0]):>4}  med={int(q[1]):>4}  "
          f"Q3={int(q[2]):>4}  p90={int(q[3]):>4}  max={int(sub.max()):>5}")